In [ ]:
# import data_preprocessing (check notebook for output)
from IPython.utils import io

with io.capture_output() as captured:
    %run data_preprocessing.ipynb

In [ ]:
# imports
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score


In [ ]:
# Prepare coaches_df
coaches_df = coaches_df.merge(
    teams_df.drop(columns=['won', 'lost']),
    on=['year', 'tmID'],
    how='left'
)

## **Task c)**: Who won each of the individual awards?

The third task consists of predicting the winners of several individual WNBA awards for the test season (season 10), using data from the previous nine seasons. The awards vary widely in criteria, statistical relevance, and the types of players typically considered for each.

##### Player Awards
- **All-Star Game Most Valuable Player**
- **Defensive Player of the Year**
- **Kim Perrot Sportsmanship Award**: player who most exemplifies the ideals of sportsmanship on the court—ethical behavior, fair play and integrity.
- **Most Improved Player**
- **Most Valuable Player**
- **Rookie of the Year**
- **Sixth Woman of the Year**: most valuable player for her team coming off the bench as a substitute.
- **WNBA Finals Most Valuable Player**

##### Coach Awards

- **Coach of the Year**

Each award reflects a distinct aspect of a player (such as scoring dominance, defensive impact, development, leadership, efficiency, or consistency). Therefore, each award requires its own model, feature set, and a filtered pool of candidates.

Is is also important to note the coach award which will make use of the teams dataframe instead of the players dataframe.

Although the awards appear to form one large **multi-class classification** problem, this is incorrect. Each award has exactly one winner per season, and every player is eligible for many or no awards. Thus, the correct formulation is a set of **independent binary classification** tasks, one per award.


### Domain-Based Sampling

As mentioned, awards reflect distinct player aspects. Additionally, award prediction is extremely imbalanced: only 1 winner among hundreds of players. In order to improve model reliability and reduce noise it is important to restrict the candidates pool for each award using domain knowledge.

- **Defensive Player of the Year**: high defensive statistics per game (`spg`, `bpg` and `sbdRpg`)
- **Kim Perrot Sportsmanship Award**: exclude players with high fouls (`pfpg` and `dqpg`)
- **Most Improved Player**: high improvement and rolling features
- **Most Valuable Player** and **All-Star Game MVP**: minimum of 50% games played and 50% minutes played, high NBA PER, team win percentage above league median
- **Rookie of the Year**: the `year` must be the player's first season
- **Sixth Woman of the Year**: bench players only (low `gspg`) with significant per game statistics
- **WNBA Finals Most Valuable Player**: only include players which attended the WNBA Finals
- **Coach of the Year**: above median teams statistics, significant improvements (e.g. `delta_win_pct`), high `rank`

It is important to mention that there is no information on the All-Star game (game played between the league's top players), but we make the assumption that only MVP candidates would be All-Star contenders.

## Domain Sampling

In [ ]:
# Domain sampling -> filter award candidates

def filter_award_candidates(df, award_name,  info=False):
    candidates = df.copy()

    if award_name in ['All-Star Game Most Valuable Player','Most Valuable Player']:
        candidates = candidates[
            (candidates['GP'] >= candidates['GP'].quantile(0.25)) &
            (candidates['mpg'] >= candidates['mpg'].quantile(0.25)) & 
            (candidates['NBA_PER'] >= candidates['NBA_PER'].quantile(0.25)) & 
            (candidates['team_win_pct'] >= candidates['team_win_pct'].median())
        ]
                
    elif award_name == 'Coach of the Year':
        # delta_cols = [col for col in candidates.columns if col.startswith('delta_')]

        candidates = candidates[
            (candidates['win_pct'] >= candidates['win_pct'].median()) &
            (candidates['delta_win_pct'] >= candidates['delta_win_pct'].quantile(0.25))
        ]

    elif award_name == 'Kim Perrot Sportsmanship Award':
        # low personal fouls and dq's
        candidates = candidates[
            (candidates['pfpg'] <= candidates['pfpg'].quantile(0.25)) &
            (candidates['dqpg'] <= candidates['dqpg'].quantile(0.25))
        ]
        
    elif award_name == 'Most Improved Player':
        delta_cols = [col for col in candidates.columns if col.startswith('delta_')]
        # rolling + deltas -> low sample ~250
        # rolling3_cols = [col for col in candidates.columns if col.endswith('_rolling3')]

        candidates = candidates[
            (candidates[delta_cols].sum(axis=1) > 0) 
            # &     (candidates[rolling3_cols].ge(candidates[rolling3_cols].quantile(0.25)).all(axis=1))
        ]
        
    elif award_name == 'Rookie of the Year':
        # filter by rookie players
        first_years = candidates.groupby('playerID')['year'] \
            .min() \
            .reset_index()
            
        first_years.rename(columns={'year': 'rookie_year'}, inplace=True)
        candidates = candidates.merge(first_years, on='playerID', how='left')
        candidates = candidates[
            (candidates['year'] == candidates['rookie_year']) 
        ]
        
    elif award_name == 'Sixth Woman of the Year':
        # mostly started as sub
        candidates = candidates[
            (candidates['gspg'] < 0.5) 
            & (candidates['GP'] >= candidates['GP'].quantile(0.25))
        ]

    elif award_name == 'WNBA Finals Most Valuable Player':
        # filter by players who played in the final
        finals = series_post_df[series_post_df['round'] == 'F']
        finals_teams = pd.concat([
            finals[['year','tmIDWinner']].rename(columns={'tmIDWinner':'tmID'}),
            finals[['year','tmIDLoser']].rename(columns={'tmIDLoser':'tmID'})
        ])

        finals_players = candidates.merge(finals_teams, on=['year','tmID'], how='inner')
        
        # Filter by players who played in the playoffs
        candidates = finals_players[finals_players['PostMinutes'] > 0]

    elif award_name == 'Defensive Player of the Year':
        candidates = candidates[
            (candidates['GP'] >= candidates['GP'].quantile(0.25)) &
            (candidates['sbdRpg'] >= candidates['sbdRpg'].quantile(0.25)) &
            (candidates['bpg'] >= candidates['bpg'].quantile(0.25)) & 
            (candidates['spg'] >= candidates['spg'].quantile(0.25)) & 
            (candidates['team_win_pct'] >= candidates['team_win_pct'].median())
        ]

    else:
        return None

    if info:
        print(award_name, len(candidates))
    
    return candidates


In [ ]:
# Domain info
print("Domain sizes per award\n")
for award in awards_players_df['award'].unique():
    df = players_teams_df

    if "Coach" in award:
        df = coaches_df

    filter_award_candidates(df, award, info=True);

In [ ]:
# it is important to define which features are not to be scaled (numeric columns that are categorical)
numeric_categorical_cols = ['year', 'stint', 'playerID', 'tmID', 'confID', "divID", "lgID", 'rank', 'playoff', "firstRound","semis","finals"]

In [ ]:
# players_teams_df skewed cols
players_num_cols = [ 
    c for c in players_teams_df.select_dtypes(include=['int64', 'float64']).columns
    if c not in numeric_categorical_cols
]

players_skewness = players_teams_df[players_num_cols].skew()
players_skewed_cols = players_skewness[abs(players_skewness) > 1].index.tolist()
print(len(players_skewed_cols))

In [ ]:
# coaches_df skewed cols
coaches_num_cols = [ 
    c for c in coaches_df.select_dtypes(include=['int64', 'float64']).columns
    if c not in numeric_categorical_cols
]
coaches_skewness = coaches_df[coaches_num_cols].skew()
coaches_skewed_cols = coaches_skewness[abs(coaches_skewness) > 1].index.tolist()
print(len(coaches_skewed_cols))

In [ ]:
# log transform on skewed cols
for c in coaches_skewed_cols:
    coaches_df[c+'_log'] = np.log1p(coaches_df[c])

for c in players_skewed_cols:
    players_teams_df[c+'_log'] = np.log1p(players_teams_df[c])

In [ ]:
# StandartScaler on the remaining numeric cols
scaler = StandardScaler()

coaches_num_cols = [c for c in coaches_num_cols if c not in coaches_skewed_cols]
players_num_cols = [c for c in players_num_cols if c not in players_skewed_cols]

coaches_df[coaches_num_cols] = scaler.fit_transform(coaches_df[coaches_num_cols])
players_teams_df[players_num_cols] = scaler.fit_transform(players_teams_df[players_num_cols])

## Preparing Datasets

In [ ]:
# defining target variables
awards_target = {
'All-Star Game Most Valuable Player': "AS_MVP",
'Defensive Player of the Year': "DPY",
'Kim Perrot Sportsmanship Award': "KPS",
'Most Improved Player': "MIP",
'Most Valuable Player': "MVP",
'Rookie of the Year': "ROOKIE",
'Sixth Woman of the Year': "SIXTH",
'WNBA Finals Most Valuable Player': "FIN_MVP",
"Coach of the Year": "COACH"
}

# TODO remove this in data cleaning
awards_players_df = awards_players_df[~awards_players_df['award'].str.contains("Decade", case=False, na=False)]

In [ ]:
player_award_list = awards_players_df['award'].unique().tolist()
player_award_list.remove("Coach of the Year")

player_award_list = list(map(lambda x: awards_target[x], player_award_list))

for award in player_award_list:
    players_teams_df[award] = 0

min_year = players_teams_df['year'].min()

for _, row in awards_players_df.iterrows():
    # if row['year'] == min_year:
    #     continue
    player_mask = (
        (players_teams_df['playerID'] == row['playerID']) & 
        (players_teams_df['year'] == row['year'] )
    )
    if row['award'] != "Coach of the Year":
        n = awards_target[row['award']]
        players_teams_df.loc[player_mask, n] = 1

In [ ]:
coach_award = awards_target["Coach of the Year"]

coaches_df[coach_award] = 0

min_year = coaches_df['year'].min()

for _, row in awards_players_df.iterrows():
    # if row['year'] == min_year:
    #     continue
    mask = (
        (coaches_df['coachID'] == row['playerID']) & 
        (coaches_df['year'] == row['year'])
    )
    coaches_df.loc[mask, coach_award] = 1

## Feature selection


In [ ]:
def filter_feature_selection(
    X: pd.DataFrame,
    y: pd.Series,
    method: str = "mutual_info",
    k: int = 10,
    corr_threshold: float = 0.9
):
    """
    Performs filter-based feature selection:
    1. Removes highly correlated features
    2. Applies statistical selection (mutual information, ANOVA, chi2)
    
    Args:
        X: Feature dataframe (after domain-based pre-filtering)
        y: Target binary series
        method: "mutual_info", "anova", or "chi2"
        k: Number of top features to select
        corr_threshold: Threshold for correlation filtering
    
    Returns:
        List of selected feature names
    """
    X_filtered = X.copy()
    
    # Correlation filtering
    corr_matrix = X_filtered.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > corr_threshold)]
    X_filtered.drop(columns=to_drop, inplace=True)
    print(f"Correlation filter removed {len(to_drop)} features: {to_drop}")
    
    # Statistical filter
    if method == "mutual_info":
        scores = mutual_info_classif(X_filtered, y, discrete_features=False, random_state=42)
    elif method == "anova":
        scores, _ = f_classif(X_filtered, y)
    elif method == "chi2":
        # Chi2 requires non-negative features
        X_pos = X_filtered - X_filtered.min() + 1e-5
        scores, _ = chi2(X_pos, y)
    else:
        raise ValueError("method must be 'mutual_info', 'anova', or 'chi2'")
    
    feature_scores = pd.Series(scores, index=X_filtered.columns)
    feature_scores = feature_scores.sort_values(ascending=False)
    selected_features = feature_scores.head(k).index.tolist()
    
    print(f"Top {k} features selected using {method}: {selected_features}")
    
    return selected_features

In [ ]:
def wrapper_feature_selection(
    X: pd.DataFrame,
    y: pd.Series,
    method: str = "RFE",
    estimator_type: str = "logistic",
    n_features_to_select: int = 10,
    cv_splits: int = 5,
    scoring: str = "roc_auc"
):
    """
    Performs wrapper-based feature selection (RFE or Sequential).
    
    Args:
        X: Feature dataframe (after domain + filter selection)
        y: Target binary series
        method: "RFE" or "SFS" (sequential forward selection)
        estimator_type: "logistic" or "gbm"
        n_features_to_select: number of features to keep
        cv_splits: number of CV splits
        scoring: evaluation metric
        
    Returns:
        List of selected feature names
    """
    # Choose estimator
    if estimator_type == "logistic":
        estimator = LogisticRegression(
            solver='liblinear', class_weight='balanced', random_state=42
        )
    elif estimator_type == "gbm":
        estimator = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, random_state=42
        )
    elif estimator_type == "rf":
        estimator = RandomForestClassifier(
            n_estimators=200, max_depth=None, class_weight='balanced', random_state=42
        )
    else:
        raise ValueError("estimator_type must be 'logistic' or 'gbm'")
    
    X_scaled = X.values
    
    # Wrapper selection
    if method == "RFE":
        selector = RFE(estimator=estimator, n_features_to_select=n_features_to_select)
        selector.fit(X_scaled, y)
        selected_features = X.columns[selector.support_].tolist()
    elif method == "SFS":
        cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
        selector = SequentialFeatureSelector(
            estimator,
            n_features_to_select=n_features_to_select,
            direction='forward',
            scoring=scoring,
            cv=cv
        )
        selector.fit(X_scaled, y)
        selected_features = X.columns[selector.get_support()].tolist()
    else:
        raise ValueError("method must be 'RFE' or 'SFS'")
    
    return selected_features

## Final Pipeline

In [ ]:
def predict_award(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    model_type: str = "logistic",
    features: list = None,
    random_state: int = 42,
    return_proba: bool = False
):
    """
    Train a model to predict a binary award target and return predictions.
    
    Args:
        X_train: Training feature dataframe
        y_train: Training target series
        X_test: Test feature dataframe
        model_type: "logistic", "rf", or "gbm"
        features: List of columns to use as features (default: use all numeric columns)
        random_state: Random state for reproducibility
        
    Returns:
        Predictions for X_test (probabilities or labels)
    """
    # If feature list is provided, subset
    if features is not None:
        X_train = X_train[features]
        X_test = X_test[features]
    else:
        X_train = X_train.select_dtypes(include=['number'])
        X_test = X_test[X_train.columns]

    # Choose model
    if model_type == "logistic":
        model = LogisticRegression(
            solver='liblinear', class_weight='balanced', random_state=random_state
        )
    elif model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=200, max_depth=None, class_weight='balanced', random_state=random_state
        )
        X_train = X_train.values
        X_test = X_test.values
    elif model_type == "gbm":
        model = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, random_state=random_state
        )
        X_train = X_train.values
        X_test = X_test.values
    else:
        raise ValueError("model_type must be 'logistic', 'rf', or 'gbm'")

    # Train
    model.fit(X_train, y_train)

    # Predict
    if return_proba:
        # Return probability of class 1
        if hasattr(model, "predict_proba"):
            preds = model.predict_proba(X_test)[:, 1]
        else:
            # For some models that don’t support predict_proba
            preds = model.predict(X_test)
    else:
        preds = model.predict(X_test)

    return preds


## Predicting Winners

In [ ]:
# NOTE secalhar nao fazer isto aqui
# Defining test/training sets -> season 10 for test set
train_teams_df = teams_df[teams_df['year'] < 10]
test_teams_df = teams_df[teams_df['year'] >= 10]

train_players_df = players_teams_df[players_teams_df['year'] < 10]
test_players_df = players_teams_df[players_teams_df['year'] >= 10]

train_coaches_df = coaches_df[coaches_df['year'] < 10]
test_coaches_df = coaches_df[coaches_df['year'] >= 10]

In [ ]:
# dicts to store features used and predictions
features_per_award = {}
predictions_per_award = {}

In [ ]:
for key, award in awards_target.items():
    print(f"award = {award}")

    if award == 'COACH':
        df_train, df_test = train_coaches_df, test_coaches_df
        id_col = 'coachID'
    else:
        df_train, df_test = train_players_df, test_players_df
        id_col = 'playerID'

    # Domain-based candidate filtering
    df_candidates = filter_award_candidates(df_train, key)
    
    # Define features and target
    X_train_sample = df_candidates.select_dtypes(include=['number'])
    y_train_sample = df_candidates[award]
    
    X_train_full = df_train.select_dtypes(include=['number'])
    y_train_full = df_train[award]
    X_test = df_test.select_dtypes(include=['number'])

    # Feature Selection
    # 1) filter-based
    filtered_features = filter_feature_selection(
        X_train_sample,
        y_train_sample,
        method='mutual_info',  # values = ['anova', 'chi2', 'mutual_info']
        k=50
    )

    X_filtered = X_train_sample[filtered_features]

    # 2) wrapper-based
    features = wrapper_feature_selection(
        X_filtered,
        y_train_sample,
        method='RFE',                    # values = ['RFE', 'SFS']
        estimator_type='logistic',       # values = ['logistic', 'gbm', 'rf']
        n_features_to_select=15,   
        # cv_splits=5,
        # scoring='roc_auc'
    )

    # store selected features
    features_per_award[award] = features    

    # run prediction
    preds = predict_award(
        X_train=X_train_full,
        y_train=y_train_full,
        X_test=X_test,
        model_type='logistic',  # values = ['logistic', 'gbm', 'rf']
        features=features,
        return_proba = True
    )

    # store predictions
    predictions_per_award[award] = preds

    # add predictions to test df
    if award == 'COACH':
        test_coaches_df[f'{award}_prob'] = preds
    else:
        test_players_df[f'{award}_prob'] = preds


In [ ]:
# Predicted winners for each award
for award in awards_target.values():
    if award == 'COACH':
        top_idx = test_coaches_df[f"{award}_prob"].idxmax()
        res = test_coaches_df.loc[top_idx]
        print(f"{award} | winner = {res['coachID']} | probability = {res[f"{award}_prob"]:.3f} | is winner ? {res[award]}")
    else: 
        top_idx = test_players_df[f"{award}_prob"].idxmax()
        res = test_players_df.loc[top_idx]
        print(f"{award} | winner = {res['playerID']} | probability = {res[f"{award}_prob"]:.3f} | is winner ? {res[award]}")

In [ ]:
awards = list(awards_target.values())
n_awards = len(awards)

# Determine subplot grid size (e.g., 2 columns)
n_cols = 3
n_rows = (n_awards + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows*3))
axes = axes.flatten()

for i, award in enumerate(awards):
    if award == 'COACH':
        df = test_coaches_df
    else:
        df = test_players_df

    sns.histplot(df[f"{award}_prob"], bins=20, kde=True, ax=axes[i])
    axes[i].set_title(f"{award}")
    axes[i].set_xlabel("Probability")
    axes[i].set_ylabel("Count")

# Turn off any extra axes if n_awards < n_rows*n_cols
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Distribution of predicted probabilites per award", fontsize=16, y=1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()  # flatten in case of single row/column

metrics_summary = {}

for i, (key, award) in enumerate(awards_target.items()):
    if award == 'COACH':
        df = test_coaches_df
    else:
        df = test_players_df

    y_true = df[award].fillna(0).astype(int)
    y_pred = (df[f'{award}_prob'] >= 0.8).astype(int)

    # Store metrics
    metrics_summary[award] = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0)
    }

    # Plot confusion matrix in subplot
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f"{award}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

# Turn off any empty subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Confusion Matrices for All Awards", fontsize=16, y=1)
plt.tight_layout()
plt.show()

# Convert metrics summary to DataFrame
metrics_df = pd.DataFrame(metrics_summary).T
metrics_df

### Model Analysis

In our pipeline, **Logistic Regression** combined with **Mutual Information** feature selection gave the best results.
**Logistic Regression** is a linear model that predicts the probability of winning an award, which makes it especially suitable for rare events like season awards, where only one winner exists per year. 
Its outputs are probabilities, which allows us to rank candidates effectively rather than just classifying them as winners or non-winners.

**Mutual Information** feature selection identifies the features most strongly associated with the target, capturing both linear and nonlinear dependencies.
By focusing the model on the most informative statistics, the pipeline avoids noise from irrelevant performance metrics and increases the predicted probabilities for likely winners.

**Random Forest** and **Gradient Boosting**, while powerful for capturing complex patterns, underperformed in our case. 
Both methods tend to produce conservative probabilities for rare positive events, especially without specialized tuning for extreme class imbalance (*which has not yet been explored*). 
**Logistic Regression**, on the other hand, could leverage the selected features directly and provide clear, interpretable probabilities, explaining why it gave higher confidence predictions for the actual award winners.

### Results Analysis

> **NOTE**
> The current pipeline uses *Mutual Information Classifier* for feature selection and *Logistic Regression* as the predictive model.

**High accuracy** is misleading: Most awards are extremely rare events (one winner per season for each award). When most observations are negative (0), a model that predicts mostly 0 will have high accuracy, even if it misses all winners.

**Precision** and **Recall** are better ways to analyze the model's performance. For the awards "Defensive Player of the Year" (`DPY`), "Most Improved Player" (`MIP`), "Most Valuable Player" (`MVP`), "WNBA Finals Most Valuable Player" (`FIN_MVP`) and "Coach of the Year" (`COACH`) the model achieved perfect values, meaning it predicted the winner correctly.
But for awards "Kim Perrot Sportsmanship Award" (`KPS`), "Rookie of the Year" (`ROOKIE`) and "Sixth Woman of the Year" (`SIXTH`) the model has 0 precision and recall meaning it failed to predict any winners.

By looking into these last 3 awards we can have some clues as to why the model probably failed.
- "Kim Perrot Sportsmanship Award" rewards sportsmanship. The `players_teams.csv` has a lot of information on a player's performance on the court. But this doesn't directly have an impact on sportsmanship. Even though domain-based sampling was applied (by choosing the players with less personal and disqualifying fouls), it does not truly reflect a player's sportsmanship.
- "Sixth Woman of the Year" award rewards a player who came off the bench as a substitute, meaning fewer started games. By itself, this should be a good metric but not enough to correctly portray the top subs. This award also **only** has 3 instances in years 8, 9, and 10, making it **extremely sparse** and difficult for the model to learn meaningful patterns.
- "Rookie of the Year" award rewards the best rookie player. Domain sampling did not emphasize the most relevant performance metrics (e.g. low per-game stats) for rookies, which may reduce predictive power.

It is also important to note that further restricting the domain sample could increase the **risk of overfitting**.When the training data is too small or too narrowly selected, the model may learn patterns that are specific to those players or seasons, rather than generalizable trends, leading to overly confident predictions that do not perform well on unseen data.
